In [33]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt 
from data_agg.data_service import DataService
from lab.atr_study import AverageTrueRange, AtrType

sys.path.append(str(Path.cwd().parent))

In [34]:
raw_df = pd.read_csv('../db/raw_parque/January_MNQ.csv')
raw_df.head(10)

,Unnamed: 0,window_start,timeframe,ticker,symbol,contract_month,contract_year,transactions,close,high,low,open,session_end_date,settlement_price,dollar_volume,volume
0,0,2026-01-08 11:38:00,1min,MNQH6,MNQ,H,2026,173,25772.50,25776.25,25771.25,25775.25,2026-01-08,NaN,9794096,380
1,1,2026-01-08 11:39:00,1min,MNQH6,MNQ,H,2026,348,25771.75,25774.50,25764.50,25772.50,2026-01-08,NaN,21363125,829
2,2,2026-01-08 11:40:00,1min,MNQH6,MNQ,H,2026,222,25778.25,25780.75,25770.75,25771.75,2026-01-08,NaN,12449620,483
3,3,2026-01-08 11:41:00,1min,MNQH6,MNQ,H,2026,147,25781.00,25782.00,25776.25,25778.00,2026-01-08,NaN,7991832,310
4,4,2026-01-08 11:42:00,1min,MNQH6,MNQ,H,2026,93,25782.50,25782.75,25779.00,25781.25,2026-01-08,NaN,4821052,187
5,5,2026-01-08 11:43:00,1min,MNQH6,MNQ,H,2026,96,25778.00,25783.00,25776.50,25783.00,2026-01-08,NaN,4717575,183
6,6,2026-01-08 11:44:00,1min,MNQH6,MNQ,H,2026,114,25774.75,25778.50,25773.25,25777.75,2026-01-08,NaN,6237596,242
7,7,2026-01-08 11:45:00,1min,MNQH6,MNQ,H,2026,129,25768.25,25775.00,25767.75,25774.25,2026-01-08,NaN,7551085,293
8,8,2026-01-08 11:46:00,1min,MNQH6,MNQ,H,2026,152,25773.25,25773.50,25767.75,25768.25,2026-01-08,NaN,8143298,316
9,9,2026-01-08 11:47:00,1min,MNQH6,MNQ,H,2026,91,25776.25,25777.50,25772.50,25773.75,2026-01-08,NaN,4639503,180


Resampling the code into 5min candles


In [35]:
raw_df['window_start'] = pd.to_datetime(raw_df['window_start'])
raw_df = raw_df.set_index('window_start')

In [36]:

agg_rules = {
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum',
    'dollar_volume': 'sum',
    'session_end_date': 'last'
}



In [37]:
df_5min = raw_df.resample('5min').agg(agg_rules)


df_5min[df_5min['session_end_date'].isna()]

df_5min_clean = df_5min.dropna(subset=['session_end_date'])

df_5min_clean['session_end_date'].unique()


<StringArray>
['2026-01-08', '2026-01-09', '2026-01-12', '2026-01-13', '2026-01-14',
 '2026-01-15', '2026-01-16', '2026-01-20', '2026-01-21', '2026-01-22',
 '2026-01-23', '2026-01-26', '2026-01-27', '2026-01-28', '2026-01-29',
 '2026-01-30']
Length: 16, dtype: str

In [38]:
research = pd.DataFrame()
research['prev_close'] = df_5min_clean['close'].shift(1)


In [39]:
research

,prev_close
window_start,
2026-01-08 11:35:00,NaN
2026-01-08 11:40:00,25771.75
2026-01-08 11:45:00,25774.75
2026-01-08 11:50:00,25777.25
2026-01-08 11:55:00,25780.00
...,...
2026-01-29 23:40:00,26008.25
2026-01-29 23:45:00,26003.50
2026-01-29 23:50:00,25991.25


In [40]:
wild_atr = AverageTrueRange(14, AtrType.WLDR, df_5min)
simp_atr = AverageTrueRange(14, AtrType.SIMP, df_5min)

wild_atr_rolling = wild_atr.rolling_atr()
simp_atr_rolling = simp_atr.rolling_atr()

TypeError: 'list' object is not callable

In [ ]:
wild_atr_14_next = wild_atr_rolling.shift(-1)
simp_atr_14_next = simp_atr_rolling.shift(-1)

AttributeError: 'AverageTrueRange' object has no attribute 'shift'